# Article Classifier

In [1]:
#imports
import pandas as pd
from tqdm import tqdm
from nltk.tokenize import RegexpTokenizer
import re
from IPython.display import display, HTML

In [2]:
keywords = (
    "kunstmatige intelligentie|artificial intelligence|artificiële intelligentie|AI|generatieve AI|"
    "generatieve kunstmatige intelligentie|generatieve artificiële intelligentie|"
    "machine learning|machinaal leren|diep leren|deep learning|neurale netwerken|"
    "large language model|grote taalmodel*|LLM|chatbot*|GPT|ChatGPT|Bard|Claude|"
    "Gemini|mistral|perplexity|ollama|LLaMA|openai|anthropic|midjourney|hugging face|"
    "slimme algoritme*|automatische besluitvorming|automatisch beslissysteem|"
    "algoritmische besluitvorming|algoritme*|cognitieve technologie*|AI-technologie*|"
    "AI-systeem*|AI-toepassing*|AI-model*|spraakherkenning|beeldherkenning|"
    "computer vision|natuurlijke taalverwerking|natural language processing|NLP|robot|drones|drone|grok|xai|deepmind|azure"
)

_COMPANY_NAMES = [
    "NVIDIA", "Apple", "Microsoft", "Google", "Alphabet",
    "Meta Platforms", "Facebook", "Tesla", "Oracle",
    "Palantir", "IBM", "Adobe", "Cambricon Technologies",
    "CoreWeave", "Fermi Inc", "Dynatrace", "Tempus AI",
    "SenseTime", "Mobileye", "Aurora Innovation", "UiPath",
    "SoundHound AI", "ASML", "NXP Semiconductors",
    "BE Semiconductor Industries", "ASM International",
    "Adyen", "Just Eat Takeaway", "Booking.com", "Mollie",
    "Picnic", "TomTom", "Swapfiets", "TKH Group",
    "Ordina", "Nedap", "CM.com", "ICT Group",
    "Neways Electronics", "Ctac", "Photon Energy",
    "Almunda Professionals", "Samsung", "Huawei",
    "Sony", "LG", "Baidu", "Tencent",
    "Alibaba", "Douyin", "Cloudflare",
    "Snowflake", "Docker", "Red Hat",
    "Uber", "Bolt", "Grab", "Epic Games",
    "Unity", "Discord", "Twitter", "X"
]

def normalize_keyword(k: str) -> str:
    """Convert wildcard-like keywords into safe, precise regex patterns."""

    k = k.strip().lower()

    # algoritme → algoritme, algoritmen, algoritmes
    if k in {"algoritme", "algoritme*"}:
        return r"algoritm(?:e|en|es)"

    # chatbots
    if k in {"chatbot", "chatbot*"}:
        return r"chatbots?"

    # llm / llms
    if k in {"llm", "llm*"}:
        return r"llms?"

    # grote taalmodel / grote taalmodels
    if k in {"grote taalmodel*"}:
        return r"grote taalmodel(?:s)?"

    # intelligent(e) algoritme(n)
    if "intelligente algoritme" in k:
        return r"intelligent[e]?\s+algoritm(?:e|en|es)?"

    # slimme algoritme(n)
    if "slimme algoritme" in k:
        return r"slimm[e]?\s+algoritm(?:e|en|es)?"

    # ANY OTHER keyword with a trailing "*" should become:
    #   <base> → <base>(?:s)?  
    # But only if safe.
    if k.endswith("*"):
        base = k[:-1]
        # optional plural “s”
        return re.escape(base) + r"s?"

    return re.escape(k)

# Convert into list
keywords = keywords.strip().split('|')
set_ai_words = {k for k in keywords if k.strip()}


In [3]:
_AI_PAT = re.compile(
    r"\b(" + "|".join(normalize_keyword(k) for k in set_ai_words) + r")\b",
    re.IGNORECASE
)

# Weiwei filter
_WEIWEI_PAT = re.compile(r'\bweiwei\b', re.IGNORECASE)

# Pattern that detects the combined form "kunstmatige intelligentie (AI)" ---
_KI_AI_PAT = re.compile(r'kunstmatige\s+intelligentie\s*\(\s*ai\s*\)', re.IGNORECASE)

def _collapse_ki_ai(matches: list[str], text: str) -> list[str]:
    """
    If the text contains 'kunstmatige intelligentie (AI)', remove up to that many 'AI'
    occurrences from the match list so the pair counts as ONE hit.
    """
    n_pairs = len(_KI_AI_PAT.findall(text))
    if n_pairs == 0:
        return matches
    kept, removed = [], 0
    for m in matches:
        if m.lower() == "ai" and removed < n_pairs:
            removed += 1         # drop this 'AI' because it's part of the pair
        else:
            kept.append(m)
    return kept

# --- Remove weiwei articles before classifying ---
def _drop_weiwei_rows(df, title_col='title', body_col='body'):
    mask = (
        df[title_col].astype(str).str.contains(_WEIWEI_PAT, na=False) |
        df[body_col].astype(str).str.contains(_WEIWEI_PAT, na=False)
    )
    removed = mask.sum()
    print(f"Removed {removed} articles containing 'weiwei'.")
    return df.loc[~mask].copy()

_COMPANY_PAT = re.compile(
    r'\b(' + '|'.join(re.escape(n) for n in _COMPANY_NAMES) + r')\b',
    re.IGNORECASE
)
_COMPANY_TOKENS = {n.lower() for n in _COMPANY_NAMES}  # to compare against matched keyword tokens

# --- 3) Classification ---
def ai_classification(df, title_col='title', body_col='body'):

    # hard remove weiwei articles
    df = _drop_weiwei_rows(df, title_col, body_col)

    labels, matched_title, matched_body, matched_all = [], [], [], []
    n_hits_title_total, n_hits_body_total = [], []
    matched_companies_all = [] #store company hits (per row)

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        title = row[title_col] if pd.notna(row[title_col]) else ""
        body  = row[body_col]  if pd.notna(row[body_col])  else ""
        
        # --- collect company hits from raw text (title + body)
        comp_title = _COMPANY_PAT.findall(str(title))
        comp_body  = _COMPANY_PAT.findall(str(body))
        company_hits = sorted(set(m.lower() for m in (comp_title + comp_body)))
        matched_companies_all.append(company_hits)

        # regex matches
        title_matches = _AI_PAT.findall(str(title))
        body_matches  = _AI_PAT.findall(str(body))

        # --- collapse KI (AI) double-counts ---
        title_matches = _collapse_ki_ai(title_matches, title)
        body_matches  = _collapse_ki_ai(body_matches, body)

        # ---  ignore 'claude' if it's the ONLY match across title+body ---
        all_lower = [m.lower() for m in (title_matches + body_matches)]
        if set(all_lower) == {"claude"}:
            title_matches, body_matches = [], []
            all_lower = []

        # decision rule
        title_match = len(title_matches) >= 1
        body_match  = len(body_matches)  >= 2
        label = "yes" if (title_match or body_match) else "no"

        # --- company mention + at least one other keyword -> ai_related = yes ---
        company_present = bool(company_hits)
        other_hits = [m for m in all_lower if m not in _COMPANY_TOKENS]
        if company_present and len(other_hits) >= 1:
            label = "yes"

        labels.append(label)
        matched_title.append(sorted(set(m.lower() for m in title_matches)))
        matched_body.append(sorted(set(m.lower() for m in body_matches)))
        matched_all.append(sorted(set(m.lower() for m in (title_matches + body_matches))))
        n_hits_title_total.append(len(title_matches))
        n_hits_body_total.append(len(body_matches))

    # write back
    df['ai_related'] = labels
    df['matched_keywords_title'] = matched_title
    df['matched_keywords_body']  = matched_body
    df['matched_keywords_all']   = matched_all
    df['n_hits_title_total'] = n_hits_title_total
    df['n_hits_body_total']  = n_hits_body_total
    df['company_hits'] = matched_companies_all

    return df


In [4]:
# Load data
df = pd.read_csv('news_parsed.csv', index_col=0)

# Apply classification (this modifies df and also drops weiwei rows)
df = ai_classification(df)

print(df['ai_related'].value_counts())

Removed 0 articles containing 'weiwei'.


100%|██████████| 29091/29091 [00:50<00:00, 580.32it/s]

ai_related
no     15366
yes    13725
Name: count, dtype: int64


In [17]:
from IPython.display import display, HTML
import pandas as pd
import re

def highlight_keywords(text, keywords):
    """
    Highlights the keywords in the given text with bold, enlarged, and green font.
    'keywords' must be the ACTUAL matched words (e.g. from matched_keywords_all),
    not the raw regex patterns.
    """
    if pd.isna(text):
        return ""
    if not isinstance(keywords, (set, list)):
        raise ValueError("Keywords must be a set or list")
    if len(keywords) == 0:
        return text

    # literal match of each keyword, case-insensitive, with word boundaries
    pattern = r'\b(?:' + '|'.join(re.escape(word) for word in keywords) + r')\b'

    def replace_keyword(match):
        kw = match.group(0)
        return f'<span style="font-size:1.5em; font-weight:bold; color:green;">{kw}</span>'

    return re.sub(pattern, replace_keyword, str(text), flags=re.IGNORECASE)


def inspect_ai_related(df, num_samples=30, random_state=42):
    """
    Displays random samples with highlighted *matched* keywords and company names.
    Uses df['matched_keywords_all'] per row instead of the global keyword list.
    """

    # --- sanity checks ---
    if 'ai_related' not in df.columns:
        raise KeyError("DataFrame must contain an 'ai_related' column.")
    if not {'title', 'body'}.issubset(df.columns):
        missing = {'title', 'body'} - set(df.columns)
        raise KeyError(f"Missing required column(s): {missing}")
    if 'matched_keywords_all' not in df.columns:
        raise KeyError("DataFrame must contain 'matched_keywords_all'.")

    # --- sample ---
    n = min(num_samples, len(df))
    samples = df.sample(n=n, random_state=random_state)

    # --- display samples ---
    for idx, row in samples.iterrows():
        ai_val = row['ai_related']
        title = row['title']
        body  = row['body']

        # --- use the actually matched keywords in this row ---
        matched_keywords = row.get('matched_keywords_all', [])
        if isinstance(matched_keywords, (list, set, tuple)):
            row_terms = {str(x) for x in matched_keywords}
        else:
            # if it's a string or something weird, just wrap it
            row_terms = {str(matched_keywords)} if matched_keywords else set()

        # add company hits if present
        companies = row.get('company_hits', [])
        if isinstance(companies, (list, set, tuple)):
            row_terms.update(str(x).lower() for x in companies)

        # --- highlight ---
        highlighted_title = highlight_keywords(title, row_terms)
        highlighted_body  = highlight_keywords(body, row_terms)

        # --- header ---
        header_html = (
            f'<div style="margin:0.5em 0;">'
            f'<strong>Index:</strong> {idx} &nbsp; | &nbsp; '
            f'<strong>ai_related:</strong> {ai_val}'
        )
        if companies:
            header_html += f' &nbsp; | &nbsp; <strong>companies:</strong> {companies}'
        if matched_keywords:
            header_html += f'<br><strong>matched keywords:</strong> {matched_keywords}'
        header_html += '</div>'

        # --- display ---
        display(HTML(header_html))
        display(HTML(f'<h3 style="margin:0.2em 0;">{highlighted_title}</h3>'))
        display(HTML(f'<div style="line-height:1.5;">{highlighted_body}</div>'))
        display(HTML('<hr>'))

    print(f"Displayed {n} random articles (with row-specific matched keywords highlighted).")
    return samples


In [34]:
# # look for specific keywords in company_hits
# import ast
# import re

# keywords_of_interest = ['x']
# pattern = '|'.join([re.escape(k) for k in keywords_of_interest])

# df_filtered_keywords = df[df['company_hits'].apply(
#     lambda kws: any(
#         re.search(pattern, str(k), re.IGNORECASE) 
#         for k in (ast.literal_eval(kws) if isinstance(kws, str) else kws)
#     )
# )]
# df_filtered_keywords

In [35]:
inspect_ai_related(df, num_samples=5, random_state=42)


Displayed 5 random articles (with row-specific matched keywords highlighted).


,Unnamed: 0,title,outlet,date,authors,body,word_count,ai_related,matched_keywords_title,matched_keywords_body,matched_keywords_all,n_hits_title_total,n_hits_body_total,company_hits
17844,17844,Pijnlijke docu stoot Adolescence van de troon ...,AD,2025-04-10,Merel van Baal,Tim Hofman benoemde het al in zijn aflevering ...,599,no,[],[algoritme],[algoritme],0,1,[]
3463,3463,Aangepast plaatsingssysteem bevalt leerlingen ...,Parool,2016-09-06,PRISCILLA TIENKAMP,"Ouders, leerlingen en onderwijsinstellingen zi...",254,yes,[],[algoritme],[algoritme],0,4,[]
2427,2427,'Academische boycot kan Israël hard in de port...,TR,2024-07-09,MERIJN VAN NULAND,De Israëlische economie komt in zwaar weer doo...,217,no,[],[kunstmatige intelligentie],[kunstmatige intelligentie],0,1,[]
3408,3408,'Tol Duitsland discrimineert buitenlanders' 'T...,Parool,2015-06-02,ANTOINE VERBIJ,Analyse: 'Infrastructuurtoeslag' blijft omstre...,235,no,[],[],[],0,0,[]
7048,7048,Jos Collignon is anti-Brexit – dus erover teke...,VK,2019-04-12,Myrel Morskate,De Brexitsaga is al bijna drie jaar aan de gan...,1114,no,[],[],[],0,0,[facebook]


In [5]:
# Save the classified data
df.to_csv('news_classified.csv')

In [ ]:
def sample_articles(df, n=120, random_state=42):
    """
    Return a random sample with columns:
    index, title, body, ai_related, matched_keywords, company_hits
    - Uses 'matched_keywords_all' if present; falls back to 'matched_keywords'
    - If 'company_hits' is missing, fills with empty lists
    """

    # build sample
    k = min(n, len(df))
    samp = df.sample(n=k, random_state=random_state).copy()

    # select/rename columns
    cols = ['title', 'body', 'ai_related', 'company_hits', 'matched_keywords_all']
    samp = samp[cols]

    # keep the original index as a column named 'index'
    samp.insert(0, 'index', samp.index)

    return samp

# usage
sample_df = sample_articles(df, n=120, random_state=42)
sample_df.head()


,index,title,body,ai_related,company_hits,matched_keywords_all
17844,17844,Pijnlijke docu stoot Adolescence van de troon ...,Tim Hofman benoemde het al in zijn aflevering ...,no,[],[]
3463,3463,Aangepast plaatsingssysteem bevalt leerlingen ...,"Ouders, leerlingen en onderwijsinstellingen zi...",no,[],[]
2427,2427,'Academische boycot kan Israël hard in de port...,De Israëlische economie komt in zwaar weer doo...,no,[],[kunstmatige intelligentie]
3408,3408,'Tol Duitsland discrimineert buitenlanders' 'T...,Analyse: 'Infrastructuurtoeslag' blijft omstre...,no,[],[]
7048,7048,Jos Collignon is anti-Brexit – dus erover teke...,De Brexitsaga is al bijna drie jaar aan de gan...,no,[facebook],[]


In [62]:
# save the sample
sample_df.to_csv('article_sample.csv', index=False)